# Medical CNN (adapted from satellite_CNN.ipynb)

This notebook reuses the model, augmentation and training pipeline from `satellite_CNN.ipynb` and adapts it to the `datasets/skin_cancer` dataset.

Cells are organized so you can run top-to-bottom: data preparation, generators, model helpers, short sanity training, and evaluation.

If image filenames use an extension other than `.jpg`, update the filename construction in cell 2.

In [ ]:
# Cell 2: imports and paths
import os
import shutil
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import PIL

# Paths - change BASE if your workspace is elsewhere
BASE = r'C:\Users\Pc\Desktop\deep_learning'
SKIN_ROOT = os.path.join(BASE, 'datasets', 'skin_cancer')
IMAGES_DIR = os.path.join(SKIN_ROOT, 'train-image', 'image')
METADATA = os.path.join(SKIN_ROOT, 'train-metadata.csv')
# Working train/test directories we will create
TRAIN_DIR = os.path.join(SKIN_ROOT, 'working_medical_cnn', 'training')
TEST_DIR = os.path.join(SKIN_ROOT, 'working_medical_cnn', 'testing')
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)
print('images dir:', IMAGES_DIR)
print('metadata:', METADATA)

In [ ]:
# Cell 3: read metadata and split files into train/test folders (stratified)
df = pd.read_csv(METADATA)
print('rows in metadata:', len(df))
# Build filename column - adjust extension if needed
df['filename'] = df['isic_id'].astype(str) + '.jpg'
label_map = {0: 'benign', 1: 'malignant'}
df['label'] = df['target'].map(label_map)
df['fullpath'] = df['filename'].apply(lambda x: os.path.join(IMAGES_DIR, x))
# keep only files that actually exist
df = df[df['fullpath'].apply(os.path.exists)].reset_index(drop=True)
print('images available after existence check:', len(df))

from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(df['fullpath'], df['label']):
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)

def copy_to(df_rows, dest_root):
    for label in df_rows['label'].unique():
        os.makedirs(os.path.join(dest_root, label), exist_ok=True)
    for _, r in tqdm(df_rows.iterrows(), total=len(df_rows)):
        src = r['fullpath']
        dst = os.path.join(dest_root, r['label'], os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

print('copying training files...')
copy_to(df_train, TRAIN_DIR)
print('copying testing files...')
copy_to(df_test, TEST_DIR)

print('train samples per class:')
for l in ['benign', 'malignant']:
    p = os.path.join(TRAIN_DIR, l)
    print(l, len(os.listdir(p)) if os.path.isdir(p) else 0)

In [ ]:
# Cell 4: ImageDataGenerators and directory iterators (augmentation similar to satellite notebook)
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE = 32
INPUT_SHAPE = (224, 224, 3)
NUM_CLASSES = 2
CLASS_MODE = 'categorical'
VALIDATION_SPLIT = 0.1

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    vertical_flip=False,
    validation_split=VALIDATION_SPLIT,
)

train_generator = train_gen.flow_from_directory(
    directory=TRAIN_DIR,
    target_size=INPUT_SHAPE[:2],
    batch_size=BATCH_SIZE,
    class_mode=CLASS_MODE,
    subset='training',
    color_mode='rgb',
    shuffle=True,
    seed=42,
)

valid_generator = train_gen.flow_from_directory(
    directory=TRAIN_DIR,
    target_size=INPUT_SHAPE[:2],
    batch_size=BATCH_SIZE,
    class_mode=CLASS_MODE,
    subset='validation',
    color_mode='rgb',
    shuffle=False,
    seed=42,
)

test_gen = ImageDataGenerator(rescale=1./255)
test_generator = test_gen.flow_from_directory(
    directory=TEST_DIR,
    target_size=INPUT_SHAPE[:2],
    batch_size=BATCH_SIZE,
    class_mode=CLASS_MODE,
    color_mode='rgb',
    shuffle=False,
    seed=42,
)

print('class indices:', train_generator.class_indices)

In [ ]:
# Cell 5: model helpers (compile_model, plot_history, display_results)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, fbeta_score, accuracy_score

def compile_model(cnn_base, input_shape, n_classes, optimizer, fine_tune=None):
    if cnn_base == 'ResNet50':
        conv_base = ResNet50(include_top=False, weights='imagenet', input_shape=input_shape)
        top_model = conv_base.output
        top_model = Flatten()(top_model)
        top_model = Dense(1024, activation='relu')(top_model)
        top_model = Dropout(0.3)(top_model)
    output_layer = Dense(n_classes, activation='softmax')(top_model)
    model = Model(inputs=conv_base.input, outputs=output_layer)
    if type(fine_tune) == int:
        for layer in conv_base.layers[fine_tune:]:
            layer.trainable = True
    else:
        for layer in conv_base.layers:
            layer.trainable = False
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['categorical_accuracy'])
    return model

def plot_history(history):
    acc = history.history.get('categorical_accuracy', [])
    val_acc = history.history.get('val_categorical_accuracy', [])
    loss = history.history.get('loss', [])
    val_loss = history.history.get('val_loss', [])
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(acc); plt.plot(val_acc)
    plt.title('accuracy'); plt.legend(['train','val'])
    plt.subplot(1,2,2)
    plt.plot(loss); plt.plot(val_loss)
    plt.title('loss'); plt.legend(['train','val'])
    plt.show()

def display_results(y_true, y_preds, class_labels):
    results = pd.DataFrame(precision_recall_fscore_support(y_true, y_preds), columns=class_labels).T
    results.rename(columns={0:'Precision',1:'Recall',2:'F-Score',3:'Support'}, inplace=True)
    conf_mat = pd.DataFrame(confusion_matrix(y_true, y_preds), columns=class_labels, index=class_labels)
    f2 = fbeta_score(y_true, y_preds, beta=2, average='micro')
    accuracy = accuracy_score(y_true, y_preds)
    print(f'Accuracy: {accuracy}')
    print(f'Global F2 Score: {f2}')
    return results, conf_mat

In [ ]:
# Cell 6: GPU config, callbacks and short sanity training run
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  try:
    tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
    print('using GPU')
  except RuntimeError as e:
    print(e)

checkpoint = ModelCheckpoint(filepath=os.path.join('model.weights.best.keras'), monitor='val_categorical_accuracy', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_categorical_accuracy', patience=6, restore_best_weights=True, mode='max')
reduce_lr = ReduceLROnPlateau(monitor='val_categorical_accuracy', factor=0.5, patience=3, min_lr=1e-6)

N_STEPS = max(1, train_generator.samples // BATCH_SIZE)
N_VAL_STEPS = max(1, valid_generator.samples // BATCH_SIZE)
N_EPOCHS = 3  # short run for sanity; increase for full training

model = compile_model('ResNet50', INPUT_SHAPE, NUM_CLASSES, Adam(learning_rate=1e-4), fine_tune=None)
model.summary()

history = model.fit(
    train_generator,
    steps_per_epoch=N_STEPS,
    epochs=N_EPOCHS,
    validation_data=valid_generator,
    validation_steps=N_VAL_STEPS,
    callbacks=[early_stop, checkpoint, reduce_lr],
)
plot_history(history)

In [ ]:
# Cell 7: Evaluation on test set (assemble images into array and predict)
model.load_weights('model.weights.best.keras')
test_paths = []
for cls in os.listdir(TEST_DIR):
    cls_dir = os.path.join(TEST_DIR, cls)
    if os.path.isdir(cls_dir):
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.jpg','.png','.jpeg')):
                test_paths.append((os.path.join(cls_dir,f), cls))

X_list = []
y_true = []
from tensorflow.keras.preprocessing.image import load_img, img_to_array
for p, cls in test_paths:
    img = load_img(p, target_size=INPUT_SHAPE[:2])
    arr = img_to_array(img)/255.0
    X_list.append(arr)
    y_true.append(0 if cls=='benign' else 1)
if len(X_list) > 0:
    X_test = np.stack(X_list)
    preds = model.predict(X_test, batch_size=32, verbose=1)
    pred_classes = np.argmax(preds, axis=1)
    class_labels = ['benign','malignant']
    prf, conf = display_results(y_true, pred_classes, class_labels)
    prf
else:
    print('no test images found; make sure the copy step completed and files exist')